In [36]:
%load_ext autoreload
%autoreload 2

In [37]:
import numpy as np
import polars as pl
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV, cross_validate, cross_val_predict, train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, roc_auc_score, make_scorer

import seaborn as sn
import matplotlib.pyplot as plt

In [2]:
# read tsv clinical matrix
clinicalMatrix = pd.read_csv("brca_data/TCGA.BRCA.sampleMap-BRCA_clinicalMatrix.tsv", sep="\t")

In [2]:
file_path = 'MOGLAM/BRCA_split/BRCA/1_featname.csv'  # Replace 'your_file.txt' with the actual file path

# Open the file and read lines into a list
with open(file_path, 'r') as file:
    file_content = file.readlines()

# Optionally, strip newline characters from each line
gene_list = [line.strip().split('|')[0] for line in file_content][1:]

# Now, file_content is a list where each element corresponds to a line in the file
print(gene_list)

['A2ML1', 'ABAT', 'ABCA13', 'ABCC11', 'ABCC8', 'ABCG1', 'ABCG2', 'ABLIM3', 'ABTB2', 'ACADSB', 'ACE2', 'ACOT4', 'ACOX2', 'ACTL6A', 'ACTR3B', 'ACVR1B', 'ADAMTS15', 'ADCY9', 'ADD1', 'AFF3', 'AGR2', 'AGR3', 'AKR1E2', 'AKR7A3', 'ALAD', 'AMD1', 'AMY1A', 'ANKLE1', 'ANKMY1', 'ANKRA2', 'ANKRD30A', 'ANKRD30B', 'ANKRD42', 'ANKRD45', 'ANKS6', 'ANLN', 'ANP32E', 'ANXA8L2', 'ANXA9', 'APBB2', 'APH1B', 'APPL2', 'ARHGAP11A', 'ARHGEF4', 'ARL9', 'ARSG', 'ART3', 'AR', 'ASNS', 'ASTN2', 'ATAD2', 'ATL2', 'ATOH7', 'ATP1B3', 'ATP5C1', 'ATP6V1C2', 'ATP7B', 'ATP8B1', 'AURKA', 'AURKB', 'B3GNT4', 'B3GNT5', 'BAIAP2L2', 'BBOX1', 'BBS1', 'BBS4', 'BBS5', 'BCAM', 'BCAS1', 'BCL11A', 'BCL2', 'BECN1', 'BHLHE40', 'BIRC5', 'BLM', 'BPI', 'BRIX1', 'BTG2', 'BTG3', 'BUB1', 'BYSL', 'C10orf116', 'C10orf26', 'C10orf32', 'C10orf90', 'C11orf75', 'C11orf82', 'C11orf86', 'C12orf11', 'C12orf48', 'C12orf72', 'C13orf27', 'C14orf174', 'C14orf45', 'C15orf23', 'C15orf42', 'C16orf45', 'C16orf57', 'C16orf61', 'C16orf71', 'C17orf28', 'C19orf51'

In [10]:
"""
Fetch interactions for a specific gene or gene list
"""

import requests
import json
import io
import polars as pl

request_url = "https://webservice.thebiogrid.org/interactions/"
# List of genes to search for
# gene_list = ["STE11", "NMD4"]  # Yeast Genes STE11 and NMD4

# evidenceList = ["POSITIVE GENETIC", "PHENOTYPIC ENHANCEMENT"]

# These parameters can be modified to match any search criteria following
# the rules outlined in the Wiki: https://wiki.thebiogrid.org/doku.php/biogridrest
params = {
    "accesskey": "7bcc32a723cda4c96a00c69790c6c26e",
    "format": "tab3",  # Return results in TAB2 format
    "geneList": "|".join(gene_list),  # Must be | separated
    "searchNames": "true",  # Search against official names
    "includeInteractors": "false",  # Set to true to get any interaction involving EITHER gene, set to false to get interactions between genes
    # "taxId": 9606,  # Limit to Saccharomyces cerevisiae
    # "evidenceList": "|".join(evidenceList),  # Exclude these two evidence types
    # "includeEvidence": "false",  # If false "evidenceList" is evidence to exclude, if true "evidenceList" is evidence to show
    "includeHeader": "true",
}

# Additional options to try, you can uncomment them as necessary
# params["start"] = 5 # Specify where to start fetching results from if > 10,000 results being returned
# params["max"] = 10 # Specify the number of results to return, max is 10,000
# params["interSpeciesExcluded"] = "false" # true or false, If ‘true’, interactions with interactors from different species will be excluded (ex. no Human -> Mouse interactions)
# params["selfInteractionsExcluded"] = "false" # true or false, If ‘true’, interactions with one interactor will be excluded. (ex. no STE11 -> STE11 interactions)
# params["searchIds"] = "false" # true or false, If ‘true’, ENTREZ_GENE, ORDERED LOCUS and SYSTEMATIC_NAME (orf) will be examined for a match with the geneList
# params["searchSynonyms"] = "false" # true or false, If ‘true’, SYNONYMS will be examined for a match with the geneList
# params["searchBiogridIds"] = "false" # true or false, If ‘true’, BIOGRID INTERNAL IDS will be examined for a match with the geneList
# params["excludeGenes"] = "false" # true or false, If 'true' the geneList becomes a list of genes to EXCLUDE rather than to INCLUDE
# params["includeInteractorInteractions"] = "true" # true or false, If ‘true’ interactions between the geneList’s first order interactors will be included. Ignored if includeInteractors is ‘false’ or if excludeGenes is set to ‘true’.
# params["htpThreshold"] = 50 # Any publication with more than this many interactions will be excluded
# params["throughputTag"] = "any" # any, low, high. If set to low, only `low throughput` interactions will be returned, if set to high, only `high throughput` interactions will be returned
# params["additionalIdentifierTypes"] = "SGD|FLYBASE|REFSEQ" # You can specify a | separated list of additional identifier types to search against (see get_identifier_types.py)

r = requests.get(request_url, params=params)
interactions = r.text
i_df = pl.read_csv(
    io.StringIO(interactions),
    separator="\t"
)
i_df.head()

#BioGRID Interaction ID,Entrez Gene Interactor A,Entrez Gene Interactor B,BioGRID ID Interactor A,BioGRID ID Interactor B,Systematic Name Interactor A,Systematic Name Interactor B,Official Symbol Interactor A,Official Symbol Interactor B,Synonyms Interactor A,Synonyms Interactor B,Experimental System,Experimental System Type,Author,Pubmed ID,Organism Interactor A,Organism Interactor B,Throughput,Score,Modification,Phenotypes,Qualifications,Tags,Source Database
i64,i64,i64,i64,i64,str,str,str,str,str,str,str,str,str,i64,i64,i64,str,str,str,str,str,str,str
143645,851076,851076,31623,31623,"""YLR362W""","""YLR362W""","""STE11""","""STE11""","""mitogen-activa…","""mitogen-activa…","""Affinity Captu…","""physical""","""Grimshaw SJ (2…",14573615,559292,559292,"""Low Throughput…","""-""","""-""","""-""","""-""","""-""","""BIOGRID"""
146985,851076,851076,31623,31623,"""YLR362W""","""YLR362W""","""STE11""","""STE11""","""mitogen-activa…","""mitogen-activa…","""Two-hybrid""","""physical""","""Printen JA (19…",7851759,559292,559292,"""Low Throughput…","""-""","""-""","""-""","""-""","""-""","""BIOGRID"""
147029,851076,851076,31623,31623,"""YLR362W""","""YLR362W""","""STE11""","""STE11""","""mitogen-activa…","""mitogen-activa…","""Reconstituted …","""physical""","""Kwan JJ (2004)…",15327964,559292,559292,"""Low Throughput…","""-""","""-""","""-""","""-""","""-""","""BIOGRID"""
147867,851076,851076,31623,31623,"""YLR362W""","""YLR362W""","""STE11""","""STE11""","""mitogen-activa…","""mitogen-activa…","""Reconstituted …","""physical""","""Bhattacharjya …",15544813,559292,559292,"""Low Throughput…","""-""","""-""","""-""","""-""","""-""","""BIOGRID"""
147868,851076,851076,31623,31623,"""YLR362W""","""YLR362W""","""STE11""","""STE11""","""mitogen-activa…","""mitogen-activa…","""Reconstituted …","""physical""","""Bhattacharjya …",15689513,559292,559292,"""Low Throughput…","""-""","""-""","""-""","""-""","""-""","""BIOGRID"""


In [ ]:
# write interactions to file
with open('test_interactions.tsv', 'w') as f:
    f.write(interactions)

In [8]:
# print(interactions)
# int_df = pl.read_csv(interactions, separator='\t')

ComputeError: invalid glob pattern given

# Class counts

In [16]:
# get two columns, drop nan and reindex from 0
cM = clinicalMatrix[['sampleID', 'PAM50_mRNA_nature2012']].dropna().reset_index(drop=True)
# print(cM)

# filter rows with PAM50_mRNA_nature2012 == 'Normal'
cM = cM[cM['PAM50_mRNA_nature2012'] != 'Normal-like'].reset_index(drop=True)

names, counts = np.unique(cM['PAM50_mRNA_nature2012'].to_numpy(), return_counts=True)
for i, j in zip(names, counts):
    print(f"{i}: {j}")

Basal-like: 98
HER2-enriched: 58
Luminal A: 231
Luminal B: 127


In [29]:
# get two columns, drop nan and reindex from 0
cM = clinicalMatrix[['sampleID', 'PAM50Call_RNAseq']].dropna().reset_index(drop=True)
print(cM)
names, counts = np.unique(cM['PAM50Call_RNAseq'].to_numpy(), return_counts=True)
for i, j in zip(names, counts):
    print(f"{i}: {j}")

            sampleID PAM50Call_RNAseq
0    TCGA-A1-A0SB-01           Normal
1    TCGA-A1-A0SD-01             LumA
2    TCGA-A1-A0SE-01             LumA
3    TCGA-A1-A0SF-01             LumA
4    TCGA-A1-A0SG-01             LumA
..               ...              ...
951  TCGA-GM-A2DM-01             LumA
952  TCGA-GM-A2DN-01             LumA
953  TCGA-GM-A2DO-01             LumB
954  TCGA-GM-A3NY-01             LumA
955  TCGA-HN-A2NL-01            Basal

[956 rows x 2 columns]
Basal: 142
Her2: 67
LumA: 434
LumB: 194
Normal: 119


In [12]:
# read mrna brca data
mrna = pl.read_csv("BRCA/mrna.csv", has_header=False)
mrna.head()



column_1,column_2,column_3,column_4,column_5,column_6,column_7,column_8,column_9,column_10,column_11,column_12,column_13,column_14,column_15,column_16,column_17,column_18,column_19,column_20,column_21,column_22,column_23,column_24,column_25,column_26,column_27,column_28,column_29,column_30,column_31,column_32,column_33,column_34,column_35,column_36,column_37,…,column_964,column_965,column_966,column_967,column_968,column_969,column_970,column_971,column_972,column_973,column_974,column_975,column_976,column_977,column_978,column_979,column_980,column_981,column_982,column_983,column_984,column_985,column_986,column_987,column_988,column_989,column_990,column_991,column_992,column_993,column_994,column_995,column_996,column_997,column_998,column_999,column_1000
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.200966,0.179005,0.532301,0.211709,0.492744,0.188533,0.499278,0.442985,0.572766,0.404017,0.62262,0.279449,0.351111,0.439866,0.537262,0.395964,0.559874,0.408594,0.548169,0.692246,0.600471,0.569744,0.467009,0.229611,0.352253,0.587638,0.608237,0.341982,0.168171,0.426876,0.465264,0.530773,0.348184,0.459637,0.229611,0.412127,0.271998,…,0.503909,0.468065,0.488102,0.602891,0.544565,0.173773,0.323132,0.503826,0.280858,0.567501,0.437433,0.297207,0.618228,0.799086,0.569308,0.549021,0.560284,0.654394,0.562774,0.485201,0.447531,0.651068,0.541465,0.590922,0.208288,0.122255,0.552747,0.241948,0.0,0.359295,0.32555,0.279449,0.364315,0.488912,0.122566,0.423955,0.401625
0.389212,0.059248,0.681862,0.059248,0.493949,0.421094,0.555361,0.424874,0.535742,0.327593,0.586114,0.176929,0.540873,0.658002,0.557575,0.433049,0.559018,0.444559,0.564504,0.663952,0.627075,0.868058,0.709061,0.13427,0.676544,0.545603,0.504139,0.059248,0.161362,0.468868,0.477229,0.56624,0.213885,0.513768,0.189931,0.327593,0.541204,…,0.511262,0.105339,0.47942,0.595474,0.50695,0.19383,0.392982,0.509523,0.339908,0.597048,0.586484,0.116202,0.640589,0.817417,0.540255,0.540968,0.569436,0.650131,0.501376,0.141961,0.523419,0.598477,0.564293,0.563123,0.035046,0.0,0.572762,0.519921,0.0,0.364035,0.336222,0.172073,0.514563,0.621025,0.277413,0.552513,0.384913
0.36787,0.054971,0.600799,0.0,0.606323,0.413671,0.638031,0.353104,0.504318,0.412384,0.671546,0.0,0.517348,0.588159,0.539941,0.372476,0.564146,0.516831,0.58844,0.695039,0.71449,0.692244,0.710583,0.342082,0.156951,0.567729,0.536461,0.093432,0.109873,0.408281,0.475521,0.017709,0.171834,0.485773,0.054971,0.282646,0.400009,…,0.510086,0.017709,0.486666,0.619106,0.51155,0.227805,0.400513,0.511981,0.263869,0.638584,0.525167,0.044387,0.66549,0.928559,0.581566,0.556119,0.630985,0.6379,0.5134,0.104765,0.539053,0.651529,0.546929,0.57356,0.444676,0.334647,0.478808,0.412059,0.0,0.351019,0.277465,0.064305,0.471376,0.629914,0.162357,0.515021,0.366723
0.306606,0.05911,0.560979,0.077592,0.464255,0.38431,0.553625,0.334694,0.375664,0.248801,0.62689,0.105145,0.426635,0.503043,0.54042,0.422111,0.563535,0.524094,0.56473,0.677211,0.618636,0.842305,0.533587,0.204247,0.605664,0.59387,0.551097,0.0,0.077592,0.407256,0.44865,0.489915,0.293771,0.478225,0.171839,0.26105,0.423994,…,0.51476,0.200852,0.46658,0.591915,0.485006,0.236544,0.372558,0.525881,0.281003,0.650033,0.287063,0.176693,0.6379,0.903232,0.561599,0.544802,0.606037,0.646317,0.490802,0.304762,0.529502,0.63537,0.535022,0.616798,0.034954,0.077592,0.391483,0.496214,0.0,0.389909,0.248801,0.166666,0.483656,0.587933,0.134054,0.451911,0.342593
0.241916,0.035355,0.616187,0.035355,0.67508,0.543972,0.518465,0.418432,0.600719,0.313845,0.667144,0.14268,0.533984,0.318751,0.567579,0.369193,0.583759,0.573306,0.566271,0.674534,0.660356,0.686051,0.433906,0.225703,0.586573,0.600943,0.540663,0.078295,0.156149,0.394229,0.466487,0.646377,0.489221,0.437167,0.134974,0.405249,0.53655

In [18]:
mrna_headers = pl.read_csv("BRCA/mrna_featname.csv", separator="|", has_header=False)
mrna_genes = mrna_headers["column_1"].to_numpy()

In [21]:
mrna_genes.shape

(1000,)

In [25]:
mrna.var()

column_1,column_2,column_3,column_4,column_5,column_6,column_7,column_8,column_9,column_10,column_11,column_12,column_13,column_14,column_15,column_16,column_17,column_18,column_19,column_20,column_21,column_22,column_23,column_24,column_25,column_26,column_27,column_28,column_29,column_30,column_31,column_32,column_33,column_34,column_35,column_36,column_37,…,column_964,column_965,column_966,column_967,column_968,column_969,column_970,column_971,column_972,column_973,column_974,column_975,column_976,column_977,column_978,column_979,column_980,column_981,column_982,column_983,column_984,column_985,column_986,column_987,column_988,column_989,column_990,column_991,column_992,column_993,column_994,column_995,column_996,column_997,column_998,column_999,column_1000
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.008022,0.026082,0.009043,0.015763,0.034453,0.024357,0.002798,0.006827,0.009005,0.005599,0.005794,0.01577,0.003923,0.011185,0.001156,0.002219,0.00101,0.015754,0.002153,0.00062,0.016469,0.037007,0.046093,0.008239,0.021136,0.001417,0.001861,0.01389,0.005008,0.001636,0.001307,0.043951,0.02532,0.001599,0.005683,0.005202,0.011114,…,0.000605,0.030767,0.005011,0.000553,0.001837,0.008577,0.002289,0.000755,0.005193,0.002151,0.025568,0.013875,0.003075,0.006785,0.000775,0.001228,0.000902,0.001689,0.001198,0.024226,0.003064,0.001319,0.000657,0.000529,0.02148,0.010577,0.00424,0.015219,0.002978,0.002883,0.002338,0.004567,0.004485,0.006709,0.009751,0.008449,0.002148


In [32]:
num_rows, num_cols = 10, 10

data = np.random.rand(num_rows, num_cols)

# Create column names (optional, you can customize these)
column_names = [f"col{i}" for i in range(num_cols)]

# Create the DataFrame from NumPy array and column names
df = pl.DataFrame(data, schema=column_names)

# Print the DataFrame
df

col0,col1,col2,col3,col4,col5,col6,col7,col8,col9
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.878595,0.388049,0.736731,0.075935,0.963925,0.314385,0.271192,0.437236,0.83282,0.926498
0.124534,0.799943,0.870047,0.552052,0.148797,0.588907,0.706904,0.548569,0.34363,0.207802
0.184589,0.666814,0.076169,0.050483,0.767831,0.900463,0.982532,0.354327,0.335867,0.930161
0.122447,0.684323,0.379226,0.92186,0.096979,0.068069,0.420772,0.769737,0.09762,0.183642
0.621008,0.909692,0.407601,0.742202,0.400199,0.218799,0.403012,0.951785,0.668837,0.503162
0.970223,0.320617,0.29923,0.333306,0.0624,0.729415,0.1756,0.904397,0.905118,0.447846
0.225181,0.435141,0.964514,0.073433,0.126074,0.992812,0.641692,0.566753,0.040015,0.814571
0.046865,0.981277,0.327753,0.309857,0.14874,0.103252,0.545018,0.385473,0.175469,0.602734
0.443936,0.862401,0.147666,0.712932,0.574447,0.796914,0.353829,0.968414,0.303511,0.434183


In [33]:
df.var()

col0,col1,col2,col3,col4,col5,col6,col7,col8,col9
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.1146,0.069471,0.097667,0.095041,0.097797,0.120615,0.078139,0.065248,0.102952,0.082083


In [46]:
df.var().to_numpy().argsort()

array([[7, 1, 6, 9, 3, 2, 4, 8, 0, 5]])

In [53]:
df.var()

col0,col1,col2,col3,col4,col5,col6,col7,col8,col9
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.1146,0.069471,0.097667,0.095041,0.097797,0.120615,0.078139,0.065248,0.102952,0.082083


In [61]:
df[df.var().melt().sort(pl.col("value"), descending=True)[:5]["variable"]]

col5,col0,col8,col4,col2
f64,f64,f64,f64,f64
0.314385,0.878595,0.83282,0.963925,0.736731
0.588907,0.124534,0.34363,0.148797,0.870047
0.900463,0.184589,0.335867,0.767831,0.076169
0.068069,0.122447,0.09762,0.096979,0.379226
0.218799,0.621008,0.668837,0.400199,0.407601
0.729415,0.970223,0.905118,0.0624,0.29923
0.992812,0.225181,0.040015,0.126074,0.964514
0.103252,0.046865,0.175469,0.14874,0.327753
0.796914,0.443936,0.303511,0.574447,0.147666


In [63]:
from bipartite_gnn.preprocessing import FeatureSelection

fs = FeatureSelection(df, 5, 0)

In [64]:
fs.variatonal_selection()

col5,col0,col8,col4,col2
f64,f64,f64,f64,f64
0.314385,0.878595,0.83282,0.963925,0.736731
0.588907,0.124534,0.34363,0.148797,0.870047
0.900463,0.184589,0.335867,0.767831,0.076169
0.068069,0.122447,0.09762,0.096979,0.379226
0.218799,0.621008,0.668837,0.400199,0.407601
0.729415,0.970223,0.905118,0.0624,0.29923
0.992812,0.225181,0.040015,0.126074,0.964514
0.103252,0.046865,0.175469,0.14874,0.327753
0.796914,0.443936,0.303511,0.574447,0.147666


In [66]:
# Assuming self.data is your DataFrame and self.n_features is the number of features you want to select
top_features = df.var().top_k(5, by="value")["variable"]

# Select the top features from the original DataFrame
selected_data = df.select(top_features)

selected_data

ColumnNotFoundError: value